[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs/Lab_2_Lagrange_Method.ipynb)


In [ ]:
# 🚀 GOOGLE COLAB SETUP
# If you are running this notebook on Google Colab, execute this cell to clone the repo and set paths:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] Google Colab setup complete! Connected to PDE-Rover-Engine repository.')
except NameError:
    pass


# 🛞 Lab 2: Quasi-Linear PDEs & Lagrange's Method
### *Mission Control: The Hands-Free Steering Wheel*
**Course:** Applied Partial Differential Equations & Planetary Robotics  
**Syllabus Topic:** Chapter 1, Section 1.3.1 (Pages 11–17)

---
### 🧭 The Rosetta Stone: Translating Math into English and Code
| 1. Professor's Blackboard Math | 2. Plain Human English | 3. Python Code |
| :--- | :--- | :--- |
| $P p + Q q = R$ | "Information moves across the terrain with velocity $(P, Q)$." | `u, v = -grad_phi_x, -grad_phi_y` |
| $\frac{dx}{P} = \frac{dy}{Q} = \frac{dz}{R}$ | "Follow the current to trace the path line." | `x += dt * P; y += dt * Q` |
| $\Phi(u, v) = 0$ | "The trajectory curves that connect start to goal." | `path = [step for step in trajectory]` |

---
## 1. In Plain English: "Riding the River Current"
Imagine you are floating in a canoe on a winding river. If you stop paddling, you don't move randomly; the river current pulls you along a specific line called a **streamline**.

A quasi-linear PDE describes how a physical quantity (like fluid flow, heat, or a rover's downhill velocity) travels.  
**Lagrange's Method of Characteristics** says:  
> *"Instead of trying to solve the entire 2D map at once, find the current lines and ride them!"*

---
## 2. The "WHY": Why do we do this method? What does it help with?
* **Why do we do it?** Solving a PDE with partial derivatives in both $x$ and $y$ simultaneously is hard.
* **What it helps with:** Lagrange transforms 1 difficult PDE into **two simple Ordinary Differential Equations (ODEs)**. It gives us the exact physical trajectory curves of the rover!


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab2

%matplotlib inline
print("Lab 2 Environment Initialized! Ready for Mission Control.")


---
## 3. The Mathematical Recipe & Worked Textbook Example

### The Working Rule (Page 11-12 of your lecture notes):
1. Put the given quasi-linear PDE in standard form: $P p + Q q = R$.
2. Write down **Lagrange's auxiliary equations**:
   $$\frac{dx}{P} = \frac{dy}{Q} = \frac{dz}{R}$$
3. Integrate two independent pairs of fractions to find two constants: $u(x, y, z) = c_1$ and $v(x, y, z) = c_2$.
4. Form the general solution: $\Phi(u, v) = 0$.

### Worked Example (Page 12-13):
$$2p - 3q = 2x$$
* **Auxiliary System:** $\frac{dx}{2} = \frac{dy}{-3} = \frac{dz}{2x}$
* **Pair 1:** $\frac{dx}{2} = \frac{dy}{-3} \implies -3dx - 2dy = 0 \implies 3x + 2y = c_1$
* **Pair 2:** $\frac{dx}{2} = \frac{dz}{2x} \implies x dx = dz \implies \frac{x^2}{2} - z = c_2$
* **General Solution:** $\Phi(3x + 2y, \frac{x^2}{2} - z) = 0$.


---
## 4. 📝 Paper & Pen Challenge 1 (Textbook Problem)

> 💡 **Instructions:** Grab your pen and paper!  
> *(If you get stuck, open `INSTRUCTOR_ANSWER_KEY.md` Section 2.1 for full steps!)*

**Problem:** Solve Euler's homogeneous equation on paper:
$$x p + y q = z$$
1. Set up the auxiliary equations: $\frac{dx}{x} = \frac{dy}{y} = \frac{dz}{z}$.
2. Pair 1 ($\frac{dx}{x} = \frac{dy}{y}$): Integrate to find constant $u(x, y)$.
3. Pair 2 ($\frac{dy}{y} = \frac{dz}{z}$): Integrate to find constant $v(y, z)$.

Enter your derived formulas for $u$ and $v$ (e.g. `"x/y"` and `"y/z"`) below:


In [ ]:
# GRADED CELL: Paper Challenge 1
### START CODE HERE ### (≈ 2 lines of code)
# Enter the two invariants u and v from your paper integration
student_u = None  # e.g. "x/y"
student_v = None  # e.g. "y/z"
### END CODE HERE ###

# Run verification test:
test_lab2.check_paper_challenge_1(student_u, student_v)


---
## 5. 🚀 The Rover Bridge Challenge (Rover Streamline Trajectories)

Now, let's connect this to the **Rover Engine**!

Suppose the rover is driving on a potential field $\phi(x, y) = x^2 + 2y^2$.  
The downhill velocity vector is $\mathbf{v} = (u, v) = -\nabla\phi = (-2x, -4y)$.  
The streamline trajectory follows the transport equation:
$$-2x \frac{\partial \phi}{\partial x} - 4y \frac{\partial \phi}{\partial y} = 0$$

**On your paper:**
1. Write Lagrange's auxiliary equation: $\frac{dx}{-2x} = \frac{dy}{-4y}$.
2. Simplify: $\frac{dx}{x} = \frac{dy}{2y}$.
3. Integrate both sides: $\ln |x| = \frac{1}{2}\ln |y| + C \implies y = k x^2$.

What curve does the rover follow? Enter the formula below:


In [ ]:
# GRADED CELL: Rover Streamline Challenge
### START CODE HERE ### (≈ 1 line of code)
# Enter the streamline trajectory equation (e.g. "y = k*x^2")
rover_streamline = None
### END CODE HERE ###

# Run verification test:
test_lab2.check_paper_challenge_2(rover_streamline)


---
## 6. 💻 Python Implementation: Numerical Characteristic Tracer

Let's code an autonomous streamline tracer using the 4th-order Runge-Kutta (RK4) ODE method to follow $(P, Q)$ vectors!


In [ ]:
# GRADED FUNCTION: trace_streamline_rk4
def trace_streamline_rk4(velocity_fn, start=(1.0, 2.0), dt=0.01, steps=50):
    # Numerically traces a characteristic curve (streamline) of a quasi-linear system
    trajectory = [start]
    curr = np.array(start, dtype=float)
    
    ### START CODE HERE ### (≈ 4-5 lines of code)
    # Loop over steps:
    # 1. vx, vy = velocity_fn(curr[0], curr[1])
    # 2. v = np.array([vx, vy], dtype=float)
    # 3. curr = curr + dt * v
    # 4. trajectory.append(tuple(curr))
    pass
    ### END CODE HERE ###
    
    return np.array(trajectory)

# Run automated unit test:
test_lab2.test_rk4_streamline_tracer(trace_streamline_rk4)


---
## 7. 🎨 Visualizing the Physics: Quiver Velocity Field & Tire Tracks

Let's plot the vector field and see our derived parabolic streamlines $y = k x^2$:


In [ ]:
x = np.linspace(-3, 3, 25)
y = np.linspace(0.1, 4, 25)
X, Y = np.meshgrid(x, y)

# Velocity field (-2x, -4y)
U = -2.0 * X
V = -4.0 * Y

plt.figure(figsize=(9, 6))
plt.quiver(X, Y, U, V, color='gray', alpha=0.5, label="Downhill Force Vector Field")

# Trace 3 characteristic curves starting from different points
def vel(x, y):
    return -2.0 * x, -4.0 * y

for x0 in [-2.0, -1.0, 1.0, 2.0]:
    path = trace_streamline_rk4(vel, start=(x0, 3.5), dt=0.005, steps=120)
    plt.plot(path[:, 0], path[:, 1], linewidth=2.5, label=f"Rover Track (Start x={x0})")

plt.title("Rover Characteristic Trajectories ($y = k x^2$ via Lagrange's Method)", fontsize=13)
plt.xlabel("East-West Coordinate (x)")
plt.ylabel("North-South Coordinate (y)")
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print("Lab 2 Complete! You have mastered Lagrange's Method of Characteristics.")
